In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path(r"./experiments_results/experiments_dp")

METRIC_FILENAME_PREFIX = "test_metrics_two_stage_final_12class"

KEY_METRICS = [
    "final_12class.macro_f1",
    "final_12class.balanced_acc",
    "final_12class.overall_acc",
    "keyword_on_vad_speech_true_nonbackground.macro_f1",
    "keyword_on_vad_speech_true_nonbackground.balanced_acc",
    "keyword_on_vad_speech_true_nonbackground.overall_acc",
    "vad.macro_f1",
    "vad.balanced_acc",
    "vad.overall_acc",
]

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
def clean_method_name(name: str) -> str:
    name = re.sub(r"[\s\-]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def extract_seed_and_method(folder_name: str):
    name = folder_name.strip()

    patterns = [
        r"(?i)(?:^|[_\-\s])seed[_\-\s]*(\d+)(?=$|[_\-\s])",
        r"(?i)(?:^|[_\-\s])(\d+)[_\-\s]*seed(?=$|[_\-\s])",
    ]

    for pat in patterns:
        m = re.search(pat, name)
        if m:
            seed = int(m.group(1))
            method = name[:m.start()] + name[m.end():]
            return clean_method_name(method), seed

    return clean_method_name(name), np.nan


for example in [
    "two_stage_ast_mfcc_spec_seed_67",
    "two_stage_ast_mfcc_spec_seed_267",
    "ast_checkpoint_speed_67_seed",
    "ast_checkpoint_none_mfcc_167_seed",
    "ast_checkpoint_base_67_seed",
    "ast_checkpoint_back_267_seed",
]:
    print(example, "->", extract_seed_and_method(example))


two_stage_ast_mfcc_spec_seed_67 -> ('two_stage_ast_mfcc_spec', 67)
two_stage_ast_mfcc_spec_seed_267 -> ('two_stage_ast_mfcc_spec', 267)
ast_checkpoint_speed_67_seed -> ('ast_checkpoint_speed', 67)
ast_checkpoint_none_mfcc_167_seed -> ('ast_checkpoint_none_mfcc', 167)
ast_checkpoint_base_67_seed -> ('ast_checkpoint_base', 67)
ast_checkpoint_back_267_seed -> ('ast_checkpoint_back', 267)


In [ ]:
def find_metric_files(base_dir: Path, prefix: str):
    if not base_dir.exists():
        raise FileNotFoundError(f"BASE_DIR nie istnieje: {base_dir}")

    files = [p for p in base_dir.rglob("*") if p.is_file() and p.name.startswith(prefix)]
    return sorted(files)


def read_one_metrics_file(path: Path) -> dict:
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"Pusty plik metryk: {path}")

    row = df.iloc[0].to_dict()
    return row


metric_files = find_metric_files(BASE_DIR, METRIC_FILENAME_PREFIX)
print(f"Znaleziono plików metryk: {len(metric_files)}")

for p in metric_files[:10]:
    print(p)

if len(metric_files) == 0:
    print("Nie znaleziono plików. Sprawdź BASE_DIR oraz METRIC_FILENAME_PREFIX.")


Znaleziono plików metryk: 9
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.1_seed_167\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.1_seed_267\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.1_seed_67\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.3_seed_167\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.3_seed_267\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.3_seed_67\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.5_seed_167\test_metrics_two_stage_final_12class.csv
experiments_results\experiments_dp\two_stage_ast_mfcc_none_dp_0.5_seed_267\test_metrics_two_stage_final_12class (19).csv
experiments_results\experiments_dp\two_st

In [ ]:
rows = []
errors = []

for metric_path in metric_files:
    try:
        rel_parts = metric_path.relative_to(BASE_DIR).parts
        top_folder = rel_parts[0] if len(rel_parts) > 1 else metric_path.parent.name

        method, seed = extract_seed_and_method(top_folder)
        row = read_one_metrics_file(metric_path)

        row.update({
            "method": method,
            "seed": seed,
            "run_folder": top_folder,
            "metric_file": str(metric_path),
        })
        rows.append(row)

    except Exception as e:
        errors.append({"file": str(metric_path), "error": repr(e)})

all_runs = pd.DataFrame(rows)

if errors:
    print("Błędy przy czytaniu niektórych plików:")
    display(pd.DataFrame(errors))

if all_runs.empty:
    raise RuntimeError("Nie udało się wczytać żadnych wyników.")

META_COLS = ["method", "seed", "run_folder", "metric_file"]
metric_cols = [c for c in all_runs.columns if c not in META_COLS]

for c in metric_cols:
    all_runs[c] = pd.to_numeric(all_runs[c], errors="coerce")

all_runs = all_runs[META_COLS + metric_cols]

print("Liczba wczytanych runów:", len(all_runs))
display(all_runs.sort_values(["method", "seed", "run_folder"]).head(30))


Liczba wczytanych runów: 9


,method,seed,run_folder,metric_file,vad.macro_f1,vad.balanced_acc,vad.overall_acc,vad.count,vad.pred_speech,vad.pred_background_noise,vad.true_speech,vad.true_background_noise,vad.false_positive_background_as_speech,vad.false_negative_speech_as_background,keyword_on_vad_speech_true_nonbackground.macro_f1,keyword_on_vad_speech_true_nonbackground.balanced_acc,keyword_on_vad_speech_true_nonbackground.overall_acc,keyword_on_vad_speech_true_nonbackground.count,keyword_on_vad_speech_true_nonbackground.loss,final_12class.macro_f1,final_12class.balanced_acc,final_12class.overall_acc,final_12class.count
2,two_stage_ast_mfcc_none_dp_0.1,67,two_stage_ast_mfcc_none_dp_0.1_seed_67,experiments_results\experiments_dp\two_stage_a...,0.482967,0.934113,0.934113,2823.0,2637.0,186.0,2823.0,0.0,0.0,186.0,0.963831,0.963621,0.963595,2637.0,0.157032,0.852738,0.900714,0.900106,2823.0
0,two_stage_ast_mfcc_none_dp_0.1,167,two_stage_ast_mfcc_none_dp_0.1_seed_167,experiments_results\experiments_dp\two_stage_a...,0.483156,0.934821,0.934821,2823.0,2639.0,184.0,2823.0,0.0,0.0,184.0,0.970989,0.970978,0.970822,2639.0,0.133028,0.859485,0.908162,0.907545,2823.0
1,two_stage_ast_mfcc_none_dp_0.1,267,two_stage_ast_mfcc_none_dp_0.1_seed_267,experiments_results\experiments_dp\two_stage_a...,0.482872,0.933758,0.933758,2823.0,2636.0,187.0,2823.0,0.0,0.0,187.0,0.969108,0.969065,0.968892,2636.0,0.136689,0.857249,0.905329,0.904711,2823.0
5,two_stage_ast_mfcc_none_dp_0.3,67,two_stage_ast_mfcc_none_dp_0.3_seed_67,experiments_results\experiments_dp\two_stage_a...,0.482967,0.934113,0.934113,2823.0,2637.0,186.0,2823.0,0.0,0.0,186.0,0.940988,0.940921,0.940463,2637.0,0.219039,0.832525,0.879060,0.878498,2823.0
3,two_stage_ast_mfcc_none_dp_0.3,167,two_stage_ast_mfcc_none_dp_0.3_seed_167,experiments_results\experiments_dp\two_stage_a...,0.483156,0.934821,0.934821,2823.0,2639.0,184.0,2823.0,0.0,0.0,184.0,0.935010,0.934941,0.934445,2639.0,0.226277,0.827373,0.874279,0.873539,2823.0
4,two_stage_ast_mfcc_none_dp_0.3,267,two_stage_ast_mfcc_none_dp_0.3_seed_267,experiments_results\experiments_dp\two_stage_a...,0.482872,0.933758,0.933758,2823.0,2636.0,187.0,2823.0,0.0,0.0,187.0,0.933216,0.932593,0.932094,2636.0,0.269610,0.825304,0.871253,0.870351,2823.0
8,two_stage_ast_mfcc_none_dp_0.5,67,two_stage_ast_mfcc_none_dp_0.5_seed_67,experiments_results\experiments_dp\two_stage_a...,0.482967,0.934113,0.934113,2823.0,2637.0,186.0,2823.0,0.0,0.0,186.0,0.782512,0.782550,0.783087,2637.0,0.912492,0.691562,0.733473,0.731491,2823.0
6,two_stage_ast_mfcc_none_dp_0.5,167,two_stage_ast_mfcc_none_dp_0.5_seed_167,experiments_results\experiments_dp\two_stage_a...,0.483156,0.934821,0.934821,2823.0,2639.0,184.0,2823.0,0.0,0.0,184.0,0.706087,0.708029,0.708223,2639.0,1.182954,0.623591,0.664674,0.662062,2823.0
7,two_stage_ast_mfcc_none_dp_0.5,267,two_stage_ast_mfcc_none_dp_0.5_seed_267,experiments_results\experiments_dp\two_stage_a...,0.482872,0.933758,0.933758,2823.0,2636.0,187.0,2823.0,0.0,0.0,187.0,0.843966,0.848538,0.847496,2636.0,0.676946,0.747083,0.791849,0.791357,2823.0


In [44]:
all_runs = all_runs[~all_runs["method"].str.startswith("experiments", na=False)]

In [ ]:
seed_check = (
    all_runs
    .groupby("method")
    .agg(
        n_runs=("metric_file", "count"),
        seeds=("seed", lambda s: ", ".join(str(int(x)) for x in sorted(s.dropna().unique()))),
        folders=("run_folder", lambda s: " | ".join(sorted(s.unique()))),
    )
    .reset_index()
    .sort_values("method")
)

display(seed_check)


,method,n_runs,seeds,folders
0,two_stage_ast_mfcc_none_dp_0.1,3,"67, 167, 267",two_stage_ast_mfcc_none_dp_0.1_seed_167 | two_...
1,two_stage_ast_mfcc_none_dp_0.3,3,"67, 167, 267",two_stage_ast_mfcc_none_dp_0.3_seed_167 | two_...
2,two_stage_ast_mfcc_none_dp_0.5,3,"67, 167, 267",two_stage_ast_mfcc_none_dp_0.5_seed_167 | two_...


In [ ]:
agg_parts = []

base_summary = (
    all_runs
    .groupby("method")
    .agg(
        n_runs=("metric_file", "count"),
        seeds=("seed", lambda s: ", ".join(str(int(x)) for x in sorted(s.dropna().unique()))),
    )
)

metric_summary = all_runs.groupby("method")[metric_cols].agg(["mean", "std", "min", "max"])
metric_summary.columns = [f"{metric}__{stat}" for metric, stat in metric_summary.columns]

summary = base_summary.join(metric_summary).reset_index()

main_sort_col = "final_12class.macro_f1__mean"
if main_sort_col in summary.columns:
    summary = summary.sort_values(main_sort_col, ascending=False)

print("Podsumowanie pełne, jedna linia = jedna metoda:")
display(summary)


Podsumowanie pełne, jedna linia = jedna metoda:


,method,n_runs,seeds,vad.macro_f1__mean,vad.macro_f1__std,vad.macro_f1__min,vad.macro_f1__max,vad.balanced_acc__mean,vad.balanced_acc__std,vad.balanced_acc__min,vad.balanced_acc__max,vad.overall_acc__mean,vad.overall_acc__std,vad.overall_acc__min,vad.overall_acc__max,vad.count__mean,vad.count__std,vad.count__min,vad.count__max,vad.pred_speech__mean,vad.pred_speech__std,vad.pred_speech__min,vad.pred_speech__max,vad.pred_background_noise__mean,vad.pred_background_noise__std,vad.pred_background_noise__min,vad.pred_background_noise__max,vad.true_speech__mean,vad.true_speech__std,vad.true_speech__min,vad.true_speech__max,vad.true_background_noise__mean,vad.true_background_noise__std,vad.true_background_noise__min,vad.true_background_noise__max,vad.false_positive_background_as_speech__mean,vad.false_positive_background_as_speech__std,vad.false_positive_background_as_speech__min,vad.false_positive_background_as_speech__max,vad.false_negative_speech_as_background__mean,vad.false_negative_speech_as_background__std,vad.false_negative_speech_as_background__min,vad.false_negative_speech_as_background__max,keyword_on_vad_speech_true_nonbackground.macro_f1__mean,keyword_on_vad_speech_true_nonbackground.macro_f1__std,keyword_on_vad_speech_true_nonbackground.macro_f1__min,keyword_on_vad_speech_true_nonbackground.macro_f1__max,keyword_on_vad_speech_true_nonbackground.balanced_acc__mean,keyword_on_vad_speech_true_nonbackground.balanced_acc__std,keyword_on_vad_speech_true_nonbackground.balanced_acc__min,keyword_on_vad_speech_true_nonbackground.balanced_acc__max,keyword_on_vad_speech_true_nonbackground.overall_acc__mean,keyword_on_vad_speech_true_nonbackground.overall_acc__std,keyword_on_vad_speech_true_nonbackground.overall_acc__min,keyword_on_vad_speech_true_nonbackground.overall_acc__max,keyword_on_vad_speech_true_nonbackground.count__mean,keyword_on_vad_speech_true_nonbackground.count__std,keyword_on_vad_speech_true_nonbackground.count__min,keyword_on_vad_speech_true_nonbackground.count__max,keyword_on_vad_speech_true_nonbackground.loss__mean,keyword_on_vad_speech_true_nonbackground.loss__std,keyword_on_vad_speech_true_nonbackground.loss__min,keyword_on_vad_speech_true_nonbackground.loss__max,final_12class.macro_f1__mean,final_12class.macro_f1__std,final_12class.macro_f1__min,final_12class.macro_f1__max,final_12class.balanced_acc__mean,final_12class.balanced_acc__std,final_12class.balanced_acc__min,final_12class.balanced_acc__max,final_12class.overall_acc__mean,final_12class.overall_acc__std,final_12class.overall_acc__min,final_12class.overall_acc__max,final_12class.count__mean,final_12class.count__std,final_12class.count__min,final_12class.count__max
0,two_stage_ast_mfcc_none_dp_0.1,3,"67, 167, 267",0.482999,0.000145,0.482872,0.483156,0.934231,0.000541,0.933758,0.934821,0.934231,0.000541,0.933758,0.934821,2823.0,0.0,2823.0,2823.0,2637.333333,1.527525,2636.0,2639.0,185.666667,1.527525,184.0,187.0,2823.0,0.0,2823.0,2823.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,185.666667,1.527525,184.0,187.0,0.967976,0.003711,0.963831,0.970989,0.967888,0.003817,0.963621,0.970978,0.967770,0.003742,0.963595,0.970822,2637.333333,1.527525,2636.0,2639.0,0.142250,0.012932,0.133028,0.157032,0.856490,0.003437,0.852738,0.859485,0.904735,0.003760,0.900714,0.908162,0.904121,0.003754,0.900106,0.907545,2823.0,0.0,2823.0,2823.0
1,two_stage_ast_mfcc_none_dp_0.3,3,"67, 167, 267",0.482999,0.000145,0.482872,0.483156,0.934231,0.000541,0.933758,0.934821,0.934231,0.000541,0.933758,0.934821,2823.0,0.0,2823.0,2823.0,2637.333333,1.527525,2636.0,2639.0,185.666667,1.527525,184.0,187.0,2823.0,0.0,2823.0,2823.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,185.666667,1.527525,184.0,187.0,0.936405,0.004069,0.933216,0.940988,0.936152,0.004294,0.932593,0.940921,0.935667,0.004316,0.932094,0.940463,2637.333333,1.527525,2636.0,2639.0,0.238309,0.027348,0.219039,0.269610,0.828401,0.003719,0.825304,0.832525,0.874864,0.003936,0.871253,0.879060,0.874129,0.004106,0.870351,0.878498,2823.0,0.0,2823.0,2823.0
2,two_stage_as

In [ ]:
def fmt_mean_std(mean, std, digits=4):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"

available_key_metrics = [m for m in KEY_METRICS if f"{m}__mean" in summary.columns]
missing_key_metrics = [m for m in KEY_METRICS if f"{m}__mean" not in summary.columns]

if missing_key_metrics:
    print("Pominięto metryki, których nie znaleziono w plikach:")
    for m in missing_key_metrics:
        print(" -", m)

comparison = summary[["method", "n_runs", "seeds"]].copy()

for m in available_key_metrics:
    comparison[m] = [
        fmt_mean_std(row[f"{m}__mean"], row[f"{m}__std"])
        for _, row in summary.iterrows()
    ]

sort_col = "final_12class.macro_f1__mean"
if sort_col in summary.columns:
    order = summary["method"].tolist()
    comparison["_order"] = comparison["method"].map({m: i for i, m in enumerate(order)})
    comparison = comparison.sort_values("_order").drop(columns="_order")

print("Krótkie porównanie metod: średnia ± odchylenie standardowe")
display(comparison)


Krótkie porównanie metod: średnia ± odchylenie standardowe


,method,n_runs,seeds,final_12class.macro_f1,final_12class.balanced_acc,final_12class.overall_acc,keyword_on_vad_speech_true_nonbackground.macro_f1,keyword_on_vad_speech_true_nonbackground.balanced_acc,keyword_on_vad_speech_true_nonbackground.overall_acc,vad.macro_f1,vad.balanced_acc,vad.overall_acc
0,two_stage_ast_mfcc_none_dp_0.1,3,"67, 167, 267",0.8565 ± 0.0034,0.9047 ± 0.0038,0.9041 ± 0.0038,0.9680 ± 0.0037,0.9679 ± 0.0038,0.9678 ± 0.0037,0.4830 ± 0.0001,0.9342 ± 0.0005,0.9342 ± 0.0005
1,two_stage_ast_mfcc_none_dp_0.3,3,"67, 167, 267",0.8284 ± 0.0037,0.8749 ± 0.0039,0.8741 ± 0.0041,0.9364 ± 0.0041,0.9362 ± 0.0043,0.9357 ± 0.0043,0.4830 ± 0.0001,0.9342 ± 0.0005,0.9342 ± 0.0005
2,two_stage_ast_mfcc_none_dp_0.5,3,"67, 167, 267",0.6874 ± 0.0619,0.7300 ± 0.0637,0.7283 ± 0.0647,0.7775 ± 0.0691,0.7797 ± 0.0703,0.7796 ± 0.0697,0.4830 ± 0.0001,0.9342 ± 0.0005,0.9342 ± 0.0005


In [ ]:
RANK_METRIC = "final_12class.macro_f1"

mean_col = f"{RANK_METRIC}__mean"
std_col = f"{RANK_METRIC}__std"

if mean_col not in summary.columns:
    raise KeyError(f"Nie ma metryki {RANK_METRIC}. Dostępne metryki: {metric_cols}")

ranking = summary[["method", "n_runs", "seeds", mean_col, std_col]].copy()
ranking = ranking.sort_values(mean_col, ascending=False)
ranking["rank"] = range(1, len(ranking) + 1)
ranking = ranking[["rank", "method", "n_runs", "seeds", mean_col, std_col]]

display(ranking)


,rank,method,n_runs,seeds,final_12class.macro_f1__mean,final_12class.macro_f1__std
1,1,two_stage_ast_mfcc_none_lr_0.0001_ca_True,3,"67, 167, 267",0.856490,0.003437
0,2,two_stage_ast_mfcc_none_lr_0.0001_ca_False,3,"67, 167, 267",0.847041,0.005796
4,3,two_stage_ast_mfcc_none_lr_1e_05_ca_False,3,"67, 167, 267",0.836081,0.006139
5,4,two_stage_ast_mfcc_none_lr_1e_05_ca_True,3,"67, 167, 267",0.831984,0.024132
3,5,two_stage_ast_mfcc_none_lr_0.001_ca_True,3,"67, 167, 267",0.812059,0.002635
2,6,two_stage_ast_mfcc_none_lr_0.001_ca_False,3,"67, 167, 267",0.797306,0.005950
6,7,two_stage_ast_mfcc_none_lr_1e_06_ca_False,3,"67, 167, 267",0.661397,0.030330
7,8,two_stage_ast_mfcc_none_lr_1e_06_ca_True,3,"67, 167, 267",0.458763,0.023095
